# AGORA NER Pipeline — Demos

Five self-contained demos showing why the LLM-based, community-aware pipeline produces entities that a general-purpose NER tool cannot.

Each demo runs locally from existing pipeline outputs — no API calls, no reprocessing.

| # | Demo | Key point |
|---|---|---|
| 1 | Disambiguation in action | Same word → 41 distinct officials |
| 2 | Text-in / entities-out | Auditable extraction with context fields |
| 3 | The "Secretary" problem | Surface-form collapse vs. structured resolution |
| 4 | Party asymmetry on entity types | Political signal only visible via role/legislation entities |
| 5 | Entity graph neighborhood | Relational structure, not just entity lists |

In [ ]:
import json, os, collections
from pathlib import Path

ROOT = Path("..").resolve()
MEMORY_DIR  = ROOT / "pipeline" / "agents" / "memory"
ENTITIES    = ROOT / "pipeline" / "agents" / "output" / "entities_canonicalized.jsonl"
FULLTEXT    = ROOT / "data" / "fulltext"
COSPONSOR   = ROOT / "knowledge_graph" / "graph_data" / "agora_cosponsors_long.csv"
SPONSORS    = ROOT / "knowledge_graph" / "graph_data" / "agora_with_sponsors.csv"
GRAPH       = ROOT / "pipeline" / "multiplex_graph" / "layer_3_entity.graphml"

def load_entities():
    recs = {}
    with open(ENTITIES) as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                recs[str(r["agora_id"])] = r
    return recs

print("Paths OK. Ready.")

---
## Demo 1 — Disambiguation in action

**Setup:** US legislation uses short tokens ("Secretary", "Director", "the Department") that change meaning from bill to bill. The pipeline maintains per-community memory files that resolve each token to the correct named referent, inferred from the bill's definitions section.

**The question:** How many distinct officials does "secretary" resolve to across the corpus?

In [ ]:
WORD = "secretary"   # try also: "director", "administrator", "commission", "department"

resolutions = []  # (community_id, resolution_text)
for fname in sorted(os.listdir(MEMORY_DIR)):
    if not fname.endswith("_memory.json"):
        continue
    d = json.load(open(MEMORY_DIR / fname))
    rules = d.get("disambiguation_rules", {})
    if WORD in rules:
        resolutions.append((d["community_id"], rules[WORD]))

print(f'"{WORD}" is disambiguated in {len(resolutions)} communities\n')
print(f"{'Community':<20}  Resolution")
print("-" * 80)
for cid, val in resolutions:
    print(f"  {cid:<18}  {val[:75]}")

In [ ]:
# Which cabinet departments show up in those resolutions?
dept_hits = collections.Counter()
keywords = [
    "Defense", "Energy", "Commerce", "Agriculture", "Labor",
    "Education", "State", "Treasury", "Interior", "Transportation",
    "Health", "Homeland", "Housing", "Veterans"
]
for _, val in resolutions:
    for kw in keywords:
        if kw.lower() in val.lower():
            dept_hits[kw] += 1
            break
    else:
        dept_hits["Other"] += 1

print("Resolved departments:")
for dept, cnt in dept_hits.most_common():
    bar = "█" * cnt
    print(f"  {dept:<15} {bar} {cnt}")

---
## Demo 2 — Text-in / entities-out with auditable context

**Setup:** Every extracted entity carries a `context` field — the exact substring from the source text that triggered the extraction. This makes every entity auditable: you can trace it back to the clause that defined it.

**The question:** What does doc 10 (intelligence community AI reporting requirements) look like before and after extraction?

In [ ]:
DOC_ID = "10"   # try any agora_id that has a fulltext file

# Raw text (first 600 chars)
text = (FULLTEXT / f"{DOC_ID}.txt").read_text(errors="ignore")
print("── RAW TEXT (first 600 chars) " + "─" * 45)
print(text[:600])
print()

In [ ]:
recs = load_entities()
rec = recs[DOC_ID]

FIELDS = {
    "organizations": "name",
    "offices":       "name",
    "roles":         "title",
    "legislation_refs": "name",
    "named_docs":    "name",
}

print("── EXTRACTED ENTITIES " + "─" * 53)
for field, key in FIELDS.items():
    ents = rec.get(field, [])
    if not ents:
        continue
    print(f"\n{field.upper()} ({len(ents)})")
    for e in ents:
        name    = e.get(key, "")
        context = e.get("context", "")[:90]
        acronym = f" [{e['acronym']}]" if e.get("acronym") else ""
        print(f"  • {name}{acronym}")
        if context:
            print(f"      context: \"{context}\"")

In [ ]:
# What disambiguation updates did the pipeline make while processing this doc?
updates = rec.get("disambiguation_updates", {})
if updates:
    print("── DISAMBIGUATION UPDATES APPLIED TO THIS DOC " + "─" * 28)
    for surface, resolved in updates.items():
        print(f"  \"{surface}\"  →  {resolved}")
else:
    print("No disambiguation updates logged for this doc (already resolved from community memory).")

print(f"\nModel: {rec.get('model')}  |  Chunks processed: {rec.get('chunks_processed')}  |  Tokens: {rec.get('prompt_tokens')}p / {rec.get('completion_tokens')}c")

---
## Demo 3 — The "Secretary" problem: surface collapse vs. structured resolution

**Setup:** A naïve NER tool returns the surface token "Secretary" 88 times across the corpus and puts all 88 mentions into one graph node. The pipeline resolves each occurrence to the correct cabinet official. This demo quantifies what information is destroyed by surface-form extraction.

**The question:** If you collapsed all "Secretary" mentions into one node, how many distinct policy domains would you be incorrectly merging?

In [ ]:
import csv

# Load documents.csv for taxonomy tags
docs_tags = {}
TAG_PREFIXES = ("Applications:", "Harms:", "Strategies:", "Risk factors:", "Incentives:")
with open(ROOT / "data" / "documents.csv", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    tag_cols = [c for c in reader.fieldnames if c.startswith(TAG_PREFIXES)]
    for row in reader:
        aid = row.get("AGORA ID", "")
        active = [c for c in tag_cols if row.get(c, "").strip() not in ("", "0", "FALSE", "False", "No")]
        if active:
            docs_tags[aid] = active

# For each role entity named "Secretary" (bare), find which docs it appears in and their tags
bare_secretary_docs = set()
resolved_secretaries = collections.defaultdict(set)  # resolved_title -> set of aids

recs_all = load_entities()
for aid, rec in recs_all.items():
    for role in rec.get("roles", []):
        title = role.get("title", "")
        if title.lower().strip() == "secretary":
            bare_secretary_docs.add(aid)
        elif title.lower().startswith("secretary of") or title.lower().startswith("secretary,"):
            resolved_secretaries[title].add(aid)

print(f"Bare 'Secretary' (unresolved) appears in {len(bare_secretary_docs)} docs")
print(f"Resolved Secretary roles: {len(resolved_secretaries)} distinct titles")
print()
print("Resolved Secretary titles and their doc counts:")
for title, aids in sorted(resolved_secretaries.items(), key=lambda x: -len(x[1])):
    print(f"  {len(aids):3d} docs  {title}")

In [ ]:
# What policy domains does each resolved Secretary appear in?
# Shows what information is LOST if you collapse to bare "Secretary"
print("Policy domain spread per resolved Secretary role:")
print("(top application tag per role)\n")

for title, aids in sorted(resolved_secretaries.items(), key=lambda x: -len(x[1]))[:10]:
    tag_counter = collections.Counter()
    for aid in aids:
        for tag in docs_tags.get(aid, []):
            if tag.startswith("Applications:"):
                tag_counter[tag.replace("Applications: ", "")] += 1
    top = tag_counter.most_common(3)
    top_str = ", ".join(f"{t} ({c})" for t, c in top) if top else "no app tag"
    print(f"  {title[:45]:<45}  [{top_str}]")

print()
print("─" * 70)
print(f"If collapsed to bare 'Secretary': 1 node spanning all domains above.")
print(f"Pipeline keeps them separate: {len(resolved_secretaries)} distinct role nodes.")

---
## Demo 4 — Party asymmetry visible only through pipeline entity types

**Setup:** spaCy cannot extract role titles or named policy documents — only person names. The pipeline's `roles` and `named_docs` fields surface entities that carry real partisan signal. This demo computes which entities are exclusive (or near-exclusive) to each party's sponsored bills.

**The question:** What does each party's AI agenda look like when expressed as entity types?

In [ ]:
# Build doc -> legislator party mapping
doc_parties = collections.defaultdict(set)   # aid -> set of party chars
leg_party   = {}                              # bioguide -> party

with open(SPONSORS, encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        aid   = row.get("AGORA ID", "")
        name  = row.get("Sponsor_Name", "")
        party = row.get("Sponsor_Party", "")
        bg    = row.get("Sponsor_bioguideId", "")
        if name and name != "None" and party in ("D", "R"):
            doc_parties[aid].add(party)
            leg_party[bg] = party

with open(COSPONSOR, encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        if row.get("Cosponsor_IsWithdrawn") == "True":
            continue
        aid   = row.get("AGORA ID", "")
        party = row.get("Cosponsor_Party", "")
        bg    = row.get("Cosponsor_BioguideId", "")
        if party in ("D", "R"):
            doc_parties[aid].add(party)
            leg_party[bg] = party

print(f"Docs with party signal: {len(doc_parties)}")

In [ ]:
# For each entity, count D-docs and R-docs it appears in (binary, one count per doc)
PIPELINE_FIELDS_KEY = {
    "organizations": "name",
    "offices":       "name",
    "roles":         "title",
    "legislation_refs": "name",
    "named_docs":    "name",
}

entity_party_docs = collections.defaultdict(lambda: {"D": set(), "R": set()})

for aid, rec in recs_all.items():
    parties = doc_parties.get(aid, set())
    for field, key in PIPELINE_FIELDS_KEY.items():
        for ent in rec.get(field, []):
            name = ent.get(key, "").strip()
            if not name:
                continue
            label = (name, field)
            for p in parties:
                entity_party_docs[label][p].add(aid)

# Find strongly party-skewed entities (min 10 docs on dominant side, <10% on other)
d_exclusive, r_exclusive = [], []
for (name, field), counts in entity_party_docs.items():
    d = len(counts["D"])
    r = len(counts["R"])
    total = d + r
    if total < 10:
        continue
    if d >= 10 and r / total < 0.1:
        d_exclusive.append((name, field, d, r))
    if r >= 10 and d / total < 0.1:
        r_exclusive.append((name, field, d, r))

d_exclusive.sort(key=lambda x: -x[2])
r_exclusive.sort(key=lambda x: -x[3])

print("DEMOCRAT-SKEWED ENTITIES (≥10 D docs, <10% R)\n")
print(f"  {'D':>4}  {'R':>4}  {'Type':<12}  Entity")
print("  " + "─" * 65)
for name, field, d, r in d_exclusive[:15]:
    print(f"  {d:>4}  {r:>4}  {field[:12]:<12}  {name[:55]}")

print()
print("REPUBLICAN-SKEWED ENTITIES (≥10 R docs, <10% D)\n")
print(f"  {'D':>4}  {'R':>4}  {'Type':<12}  Entity")
print("  " + "─" * 65)
for name, field, d, r in r_exclusive[:15]:
    print(f"  {d:>4}  {r:>4}  {field[:12]:<12}  {name[:55]}")

In [ ]:
# Spotlight: "Developer" and "Deployer" role titles — purely Democratic AI governance concepts
for target in ["Developer", "Deployer"]:
    counts = entity_party_docs.get((target, "roles"), {"D": set(), "R": set()})
    d, r = len(counts["D"]), len(counts["R"])
    print(f'Role title "{target}": D={d} docs  R={r} docs')
    # Show one example context
    for aid, rec in recs_all.items():
        for role in rec.get("roles", []):
            if role.get("title","").strip() == target:
                print(f'  example context: "{role.get("context","")[:100]}"')
                break
        else:
            continue
        break
    print()

---
## Demo 5 — Entity graph neighborhood: one bill as a structured object

**Setup:** The pipeline produces a knowledge graph (`layer_3_entity.graphml`) where document nodes connect to every entity they reference. This demo loads the subgraph around the highest-degree document and shows what it looks like as a structured object vs. a raw text blob.

**The question:** What does document:77 — the hub with the most entity connections (230 edges) — look like as a graph neighborhood?

In [ ]:
import networkx as nx

G = nx.read_graphml(str(GRAPH))
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Find hub documents by degree
U = G.to_undirected()
doc_nodes = [(n, U.degree(n), G.nodes[n].get("label", n))
             for n, d in G.nodes(data=True) if d.get("node_type") == "Document"]
doc_nodes.sort(key=lambda x: -x[1])

print("\nTop 10 hub documents:")
print(f"  {'Degree':>6}  Node ID")
for nid, deg, label in doc_nodes[:10]:
    print(f"  {deg:>6}  {label}")

In [ ]:
HUB_DOC = "document:77"   # change to any doc node id from the list above

neighbors = list(U.neighbors(HUB_DOC))
by_type = collections.defaultdict(list)
for nb in neighbors:
    ntype = G.nodes[nb].get("node_type", "?")
    label = G.nodes[nb].get("label", nb)
    deg   = U.degree(nb)
    by_type[ntype].append((label, deg))

agora_id = HUB_DOC.replace("document:", "")
print(f"Neighborhood of {HUB_DOC}  (agora_id={agora_id})")
print(f"Total connected entities: {len(neighbors)}\n")

for ntype in ["Org", "Role", "Office", "Legislation", "Named Doc"]:
    items = sorted(by_type.get(ntype, []), key=lambda x: -x[1])
    if not items:
        continue
    print(f"{ntype.upper()} ({len(items)})")
    for label, deg in items[:12]:
        print(f"  {'·'} {label[:70]}")
    if len(items) > 12:
        print(f"    … {len(items)-12} more")
    print()

In [ ]:
# Which of those entity nodes are also connected to many OTHER documents?
# Those are the "universal" entities — the cross-cutting nodes of the graph.
print("Cross-cutting entities in this hub's neighborhood")
print("(connected to this doc AND ≥10 other docs)\n")
print(f"  {'This-doc deg':>12}  {'Graph deg':>9}  {'Type':<12}  Entity")
print("  " + "─" * 65)

cross_cutting = []
for nb in neighbors:
    ntype = G.nodes[nb].get("node_type", "?")
    if ntype == "Document":
        continue
    label   = G.nodes[nb].get("label", nb)
    g_deg   = U.degree(nb)
    # count how many document neighbors this entity has
    doc_neighbors = sum(1 for n in U.neighbors(nb) if G.nodes[n].get("node_type") == "Document")
    if doc_neighbors >= 10:
        cross_cutting.append((label, ntype, g_deg, doc_neighbors))

cross_cutting.sort(key=lambda x: -x[3])
for label, ntype, g_deg, doc_cnt in cross_cutting[:20]:
    print(f"  {doc_cnt:>12} docs  {g_deg:>9}  {ntype:<12}  {label[:50]}")

print(f"\n{len(cross_cutting)} cross-cutting entities in this neighborhood out of {len(neighbors)} total.")

---

## What these demos collectively show

| Demo | What a keyword/regex approach gives you | What the pipeline gives you |
|---|---|---|
| 1 | The string `"secretary"` — 88 hits | 41 distinct cabinet officials, each pinned to a bill section |
| 2 | A list of matching spans | Structured entities with type, acronym, parent org, and extractive context |
| 3 | One overloaded "Secretary" graph node | Separate nodes for DoD, HHS, Agriculture, Energy… each in its own policy domain |
| 4 | Bag-of-words party comparison | Role titles ("Developer", "Deployer") and legislation as partisan signals |
| 5 | A text file | A graph: 230 entity connections, typed, traversable, cross-document |

The common thread: the pipeline produces **relational structure**, not just occurrence lists. The graph edges are the output — they encode which entities co-appear in the same legislative instrument, which is the unit of analysis for AI policy research.

> **Interactive visualization:** open `reports/community_001_network.html` in a browser for a clickable version of the military community subgraph (294 nodes, 1001 edges).